# X-ray luminosity interpolator

This notebook create an interpolator function for the cooling curves obtained from magneto-thermal simulations for different values of the initial magnetic field.

In [ ]:
#import libraries
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scipy.integrate as integrate
from scipy import interpolate
from scipy.interpolate import UnivariateSpline
import pickle

import pypopsyn.simulator.basics.constants as const
from pypopsyn.simulator.config_simulator import cfg
import utilities.plot_settings

Load the results from the magneto-thermal simulations.

In [ ]:
df_B12 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e12_Btor1e13.csv",
    delimiter=",",
    header=[0],
)
df_B12.head()

In [ ]:
df_B13 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e13_Btor1e14.csv",
    delimiter=",",
    header=[0],
)
df_B13.head()

In [ ]:
df_B14 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e14_Btor1e15.csv",
    delimiter=",",
    header=[0],
)
df_B14.head()

In [ ]:
df_B15 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_B15.head()

In [ ]:
df_B5e15 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip5e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_B5e15.head()

In [ ]:
t12 = df_B12["t[yr]"].to_numpy().astype(float) 
t13 = df_B13["t[yr]"].to_numpy().astype(float) 
t14 = df_B14["t[yr]"].to_numpy().astype(float) 
t15 = df_B15["t[yr]"].to_numpy().astype(float) 
t5e15 = df_B5e15["t[yr]"].to_numpy().astype(float) 
L12 = df_B12["L[erg/s]"].to_numpy().astype(float) 
L13 = df_B13["L[erg/s]"].to_numpy().astype(float) 
L14 = df_B14["L[erg/s]"].to_numpy().astype(float) 
L15 = df_B15["L[erg/s]"].to_numpy().astype(float) 
L5e15 = df_B5e15["L[erg/s]"].to_numpy().astype(float) 

In [ ]:
log_B_range = np.array([12, 13, 14, 15, np.log10(5.e15)])

colors = plt.get_cmap("viridis", len(log_B_range))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

ax.set_xscale('log') 
ax.set_yscale('log')
ax.set_ylim(1.e29, 1.e37)
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$L_{X}$ [erg s$^{-1}$]')

ax.plot( 
    t12,
    L12,
    linestyle='-',
    linewidth=4,
    color=colors(0),
    rasterized=True,
    label=r"$B_0 = 10^{12}$ G",
)
ax.plot( 
    t13,
    L13,
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
    label=r"$B_0 = 10^{13}$ G",
)
ax.plot( 
    t14,
    L14,
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
    label=r"$B_0 = 10^{14}$ G",
)
ax.plot( 
    t15,
    L15,
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
    label=r"$B_0 = 10^{15}$ G",
)
ax.plot( 
    t5e15,
    L5e15,
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
    label=r"$B_0 = 5 \times 10^{15}$ G",
)
plt.legend(frameon=False, loc=0)
plt.grid()

Construct the interpolator function.

In [ ]:
# Since the length of the cooling curves is different, let's first interpolate them on the same time grid in [yr].
time_grid = np.logspace(0.,6., 100)

L12_interpolator = interpolate.UnivariateSpline(t12, L12, k=1)
L13_interpolator = interpolate.UnivariateSpline(t13, L13, k=1)
L14_interpolator = interpolate.UnivariateSpline(t14, L14, k=1)
L15_interpolator = interpolate.UnivariateSpline(t15, L15, k=1)
L5e15_interpolator = interpolate.UnivariateSpline(t5e15, L5e15, k=1)

L12_interp = L12_interpolator(time_grid)
L13_interp = L13_interpolator(time_grid)
L14_interp = L14_interpolator(time_grid)
L15_interp = L15_interpolator(time_grid)
L5e15_interp = L5e15_interpolator(time_grid)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

ax.set_xscale('log') 
ax.set_yscale('log')
ax.set_ylim(1.e29, 1.e37)
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$L_{X}$ [erg s$^{-1}$]')

ax.plot( 
    time_grid,
    L12_interp,
    linestyle='-',
    linewidth=4,
    color=colors(0),
    rasterized=True,
    label=r"$B_0 = 10^{12}$ G",
)
ax.plot( 
    time_grid,
    L13_interp,
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
    label=r"$B_0 = 10^{13}$ G",
)
ax.plot( 
    time_grid,
    L14_interp,
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
    label=r"$B_0 = 10^{14}$ G",
)
ax.plot( 
    time_grid,
    L15_interp,
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
    label=r"$B_0 = 10^{15}$ G",
)
ax.plot( 
    time_grid,
    L5e15_interp,
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
    label=r"$B_0 = 5 \times 10^{15}$ G",
)
plt.legend(frameon=False, loc=0)
plt.grid()

In [ ]:
# Create a grid of initial magnetic field and stack together all the cooling curves.
B0 = np.array([1.e12, 1.e13, 1.e14, 1.e15, 5.e15])
L_t_stack = np.vstack((L12_interp, L13_interp, L14_interp, L15_interp, L5e15_interp)).T

# Define the minimum and maximum age in [yr] and the minimum and maximum initial magnetic field in [G].
time_range = np.array([0.0, 1.e8])
B0_range = np.array([1.e11, 1.e17])

Lx_interpolator = interpolate.RectBivariateSpline(
    time_grid, 
    B0, 
    L_t_stack, 
    bbox=[time_range[0], time_range[1], B0_range[0], B0_range[1]], 
    kx=1, 
    ky=1
)

In [ ]:
# Save the interpolator function and try to import it again to see if it works.
with open('../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/interpolator_Lx.pkl', 'wb') as f:
    pickle.dump(Lx_interpolator, f)

with open('../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/interpolator_Lx.pkl', 'rb') as f:
    Lx_interpolator_import = pickle.load(f)

In [ ]:
# Define a grid of time and initial magnetic field where to evaluate the interpolated cooling curves.
t_eval = np.logspace(0., 6., 500)
B0_eval = np.logspace(11., 17., 30)

Lt_interp = Lx_interpolator_import(t_eval, B0_eval)
print(Lt_interp.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(B0_eval))
norm = mpl.colors.Normalize(vmin=np.min(log_B_range), vmax=np.max(log_B_range)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale('log') 
ax.set_yscale('log')
ax.set_ylim(1.e29, 1.e37)
ax.set_xlabel(r'Time [yr]')
ax.set_ylabel(r'$L_{X}$ [erg s$^{-1}$]')

ax.plot( 
    time_grid,
    L12_interp,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    time_grid,
    L13_interp,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    time_grid,
    L14_interp,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    time_grid,
    L15_interp,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)
ax.plot( 
    time_grid,
    L5e15_interp,
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
)

for i in range(len(B0_eval)):
    ax.plot( 
        t_eval,
        Lt_interp[:,i],
        linestyle='-',
        linewidth=4,
        color=colors(i),
        rasterized=True,
        alpha=0.5
    )
plt.grid()

In [ ]:
# Try to evaluate the X-ray luminosity from random values of the age and the initial magnetic field. 
t_eval_test = np.array([1.e3, 1.e2])
B0_eval_test = np.logspace(13., 15, 2)

Lt_interp_test = Lx_interpolator_import.ev(t_eval_test, B0_eval_test)
print(Lt_interp_test)